# VaR 99% de 1 dia: calibração e tempo clássico

Mede o tempo do Monte Carlo (MC) e do quasi-Monte Carlo (QMC) para a carteira de opções de `carteira.py`, com erro relativo de 1% do VaR. A saída principal continua em `../dados/tau_c.csv`.

A reta ajustada aos três valores iniciais de M é um diagnóstico; ela deixou de determinar sozinha o tamanho da amostra. O erro é efetivamente medido, entre 50 sementes, em amostras próximas do alvo. A lista completa de pontos medidos é gravada em `../dados/e1_validacao_erro.csv`.

- **MC:** a primeira proposta perto do alvo usa a taxa teórica M^(-1/2), ancorada no maior piloto medido. M pode ser qualquer inteiro, sem arredondamento artificial para potência de dois. O erro observado confirma ou ajusta a proposta.
- **QMC:** cada semente é um embaralhamento Sobol independente. O total de cenários permanece potência de dois. Depois de uma proposta inicial da reta, são medidos tamanhos vizinhos até encontrar um tamanho que atende ao alvo e um menor que não atende.
- **Controle do ajuste:** a saída registra M contínuo previsto pela reta, seu arredondamento, o fator correspondente, M finalmente executado e seu erro observado. O fator de arredondamento da previsão não deve ser confundido com a mudança decorrente da validação.

O critério é empírico: desvio padrão amostral do VaR dividido pela média entre as 50 sementes ≤ 1%. A adaptação usa essas mesmas sementes e não representa uma garantia estatística independente nem prova de mínimo global de M. O tempo com erro de 0,1% **não é medido aqui**: continua uma extrapolação em E3 e deve ser identificado dessa forma.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

import pandas as pd

DADOS = Path("../dados")
CACHE = Path(tempfile.gettempdir()) / "quant-e1-validacao"
SCRIPT = Path("e1_validacao.py").resolve()


## Validação do erro e cronometração

`e1_validacao.py` contém a rotina usada tanto pelo notebook quanto pela linha de comando. O notebook a executa em processo novo com as variáveis de threads definidas antes da importação de NumPy/SciPy. Os pilotos são M = 1.024, 4.096 e 16.384; os tamanhos adicionais dependem do erro observado. A calibração pode distribuir pares (N, método) entre processos. **A cronometração final ocorre em série**, sem tarefas de calibração concorrentes, e usa a mediana de três execuções da função original `carteira.calcular_var`, que reavalia todas as N × K opções.

Somente para a calibração, a paridade put-call reduz duas avaliações equivalentes de cada straddle a uma. A distribuição de perdas, as normais, a correlação por Cholesky e os cenários permanecem iguais. Antes da execução, a rotina confere numericamente essa identidade contra a reavaliação original para os dois métodos. Essa redução não é usada para obter os tempos da referência clássica.

Para cronometração reproduzível, execute sem outras tarefas pesadas e, na linha de comando, fixe as threads de álgebra linear, por exemplo:

```bash
VECLIB_MAXIMUM_THREADS=1 OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python codigo/e1_validacao.py
```

O cache guarda apenas medições da calibração, indexadas por N, método e número de sementes. Para medir novamente com outra versão de software ou hardware, use um diretório de cache vazio. A cronometração sempre é refeita.


In [ ]:
# O processo novo recebe o limite de threads antes de importar NumPy/SciPy,
# mesmo que o kernel do notebook já tenha importado essas bibliotecas.
env = os.environ.copy()
env.update(VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1")
processo = subprocess.Popen(
    [sys.executable, str(SCRIPT), "--dados", str(DADOS.resolve()),
     "--cache-dir", str(CACHE), "--workers", "4"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for linha in processo.stdout:
    print(linha, end="", flush=True)
if processo.wait():
    raise RuntimeError("E1 não concluiu; confira a saída acima.")


In [ ]:
tempos = pd.read_csv(DADOS / "tau_c.csv")
colunas = ["N", "metodo", "M_previsto_reta", "M_arredondado_reta",
           "fator_arredondamento_reta", "fator_potencia2_reta", "M", "erro_rel_medido",
           "M_menor_medido", "erro_rel_menor", "tau_c_s"]
display(tempos[colunas])
assert (tempos["erro_rel_medido"] <= 0.01).all()
assert (tempos["sementes"] == 50).all()
assert all((int(m) & (int(m) - 1)) == 0 for m in tempos.loc[tempos.metodo == "qmc", "M"])


## Como ler o resultado

O diagnóstico permite conferir diretamente o caso N = 32: a inclinação de três pontos continua visível, mas o tempo não depende de extrapolar essa inclinação até o alvo sem medição. Para Sobol, a potência de dois é uma exigência da amostragem; seu efeito sobre o tamanho proposto fica explícito, em vez de ser escondido na comparação dos tempos. O arredondamento para potência de dois foi removido do MC.

Os tempos usam a mesma medida empírica de erro, não necessariamente o mesmo número de cenários. A comparação com E3 usa esse tempo medido no alvo de 1%; extrapolações para 0,1% precisam permanecer separadas. Fila e rede não entram nos tempos.

A referência mantém a reavaliação completa por Black–Scholes. Ela não inclui otimização clássica por tabelas nem distribuição do cálculo entre máquinas. As inclinações e os tempos cabem nas tabelas do artigo; este notebook não exporta as figuras redundantes retiradas na revisão.
